**ENTENDIENDO LOS DATOS**

**TRATAMIENTO DE LAS ENCUESTAS ENAHO (MODULO 1).**

In [55]:
import os
import glob
import pandas as pd
import numpy as np

# ==============================================================================
# CONFIGURACIÓN DE RUTAS Y CONSTANTES
# ==============================================================================
RUTA_MODULO1 = r"D:/FUND. CIENCIA DE DATOS/CDD-2025" 
RUTA_MODULO2_2024 = r"D:/FUND. CIENCIA DE DATOS/CDD-2025/2024"
RUTA_SALIDA = r"D:/FUND. CIENCIA DE DATOS/CDD-2025/Resultados_Procesados"

os.makedirs(RUTA_SALIDA, exist_ok=True)

# Variables a filtrar en el Módulo 1 (Carátula y Ficha de la Vivienda)
VARIABLES_A_FILTRAR = ['conglome', 'vivienda', 'hogar', 'ubigeo', 'dominio', 'estrato', 'result']

**CONFIGURACIÓN GLOBAL DEL ENTORNO**

Lee archivos CSV del INEI probando codificaciones y delimitadores ( ' ; ' , ' , ' , ' \t ').

In [56]:
def cargar_csv(ruta_archivo):
    
    encodings = ['latin1', 'cp1252', 'iso-8859-1', 'utf-8']
    separadores = [';', ',', '\t']
    
    errores = []
    
    for enc in encodings:
        for sep in separadores:
            try:
                df = pd.read_csv(
                    ruta_archivo, 
                    encoding=enc, 
                    sep=sep, 
                    low_memory=False, 
                    on_bad_lines='skip'
                )
                
                # Descartar si toda la fila se leyó en una sola columna por mal delimitador
                if len(df.columns) <= 1:
                    continue
                
                df.columns = df.columns.str.lower().str.strip()
                return df
            except Exception as e:
                errores.append(f"Encoding={enc}, Sep='{sep}' -> Error: {str(e)}")
                continue
                
    detalle_error = "\n".join(errores[:3])
    raise ValueError(f"No se pudo leer el archivo: {ruta_archivo}\nDetalles:\n{detalle_error}")

**1: Alteración del Problema por Valores Faltantes en result**

Cuando la variable **result** adopta valores distintos de 1 (Completa) o 2 (Incompleta) —por ejemplo rechazo, ausente o vivienda desocupada— el encuestador no logra aplicar los módulos sociodemográficos posteriores. **Esto genera una condición de datos faltantes estructurales**.

**Impacto en la modelación:** No es posible usar características socioeconómicas (ingresos, educación, empleo) como variables predictoras de la No Respuesta, ya que estarán nulas en todos los casos donde el evento ocurra.

**Variables utilizables:** Únicamente se pueden considerar variables recolectadas al inicio de la visita o de orden geográfico/espacial (ubigeo, dominio, estrato, conglome, vivienda, hogar.).

**2: Evaluación de Utilidad del Módulo 2**

El Módulo 2 registra las características de los miembros del hogar. Al surgir una situación de rechazo o vivienda desocupada (result >= 3), no existe registro de personas en este módulo.

Lo que nos lleva a concluir que No es posible utilizar el Módulo 2 para la predicción de la no respuesta, ya que combinar (merge) el Módulo 1 con el Módulo 2 eliminaría las observaciones de no respuesta o dejaría registros totalmente vacíos en los predictores.

**3:FILTRADO DE VARIABLES**

In [57]:
# ==============================================================================
# PUNTO 3: FILTRADO DE VARIABLES VÁLIDAS
# ==============================================================================
archivos_m1 = glob.glob(os.path.join(RUTA_MODULO1, "*.csv"))

dict_dataframes_filtrados = {}
reporte_procesamiento = []

print("Procesando la recolección y filtrado de microdatos (Módulo 1)...\n")

for archivo in archivos_m1:
    nombre_archivo = os.path.basename(archivo)
    
    # Extraer el año correspondiente
    anio = next((y for y in range(2019, 2026) if str(y) in nombre_archivo), None)
    if anio is None:
        continue
    
    try:
        df = cargar_csv(archivo)
    except Exception as err:
        reporte_procesamiento.append({
            "Año": anio,
            "Registros (N)": 0,
            "Num_Cols": 0,
            "Estado": f"Error de lectura"
        })
        continue

    # Identificar columna target
    col_result = [c for c in df.columns if 'result' in c]
    if not col_result:
        reporte_procesamiento.append({
            "Año": anio,
            "Registros (N)": len(df),
            "Num_Cols": 0,
            "Estado": "Sin columna result"
        })
        continue
    target_var = col_result[0]

    # Construir lista de variables presentes en el dataset
    vars_disponibles = [v for v in VARIABLES_A_FILTRAR if v in df.columns]
    if target_var not in vars_disponibles:
        vars_disponibles.append(target_var)
        
    df_filtrado = df[vars_disponibles].copy()
    dict_dataframes_filtrados[anio] = df_filtrado
    
    # Agregar registro al reporte formal
    reporte_procesamiento.append({
        "Año": anio,
        "Registros (N)": f"{len(df_filtrado):,}",
        "N° Vars": len(df_filtrado.columns),
        "Variables Incluidas": ", ".join(df_filtrado.columns),
        "Estado": "Procesado OK"
    })

# ==============================================================================
# IMPRESIÓN DEL REPORTE TABULAR FORMAL
# ==============================================================================
df_reporte = pd.DataFrame(reporte_procesamiento).sort_values("Año")

print("=" * 85)
print("             RESUMEN : FILTRADO DE VARIABLES MÓDULO 1 (ENAHO)")
print("=" * 85)
print(df_reporte.to_string(index=False))
print("=" * 85)

Procesando la recolección y filtrado de microdatos (Módulo 1)...

             RESUMEN : FILTRADO DE VARIABLES MÓDULO 1 (ENAHO)
 Año Registros (N)  N° Vars                                         Variables Incluidas       Estado
2019        43,868        7 conglome, vivienda, hogar, ubigeo, dominio, estrato, result Procesado OK
2020        53,423        7 conglome, vivienda, hogar, ubigeo, dominio, estrato, result Procesado OK
2021        43,524        7 conglome, vivienda, hogar, ubigeo, dominio, estrato, result Procesado OK
2022        44,122        7 conglome, vivienda, hogar, ubigeo, dominio, estrato, result Procesado OK
2023        44,378        7 conglome, vivienda, hogar, ubigeo, dominio, estrato, result Procesado OK
2024        44,731        7 conglome, vivienda, hogar, ubigeo, dominio, estrato, result Procesado OK
2025        44,599        7 conglome, vivienda, hogar, ubigeo, dominio, estrato, result Procesado OK


**4:TABLA DE PORCENTAJES**

In [58]:

# ==============================================================================
# PUNTO 4: CUADRO DE PORCENTAJES DE NO RESPUESTA (RESULT) A NIVEL HOGAR
# ==============================================================================
print("--- Ejecutando Punto 4: Generación de Cuadro Temporal de No Respuesta ---")

if 'dict_dataframes_filtrados' not in globals() or not dict_dataframes_filtrados:
    raise NameError("El diccionario 'dict_dataframes_filtrados' no está en memoria. Ejecuta primero el Punto 3.")
 
resumen_resultados = []

for anio, df in sorted(dict_dataframes_filtrados.items()):
    # Identificar la columna 'result' (ignorando mayúsculas/minúsculas)
    cols_result = [c for c in df.columns if 'result' in c.lower()]
    
    if not cols_result:
        print(f"Advertencia: No se encontró la columna 'result' para el año {anio}")
        continue
        
    col_result = cols_result[0]
    
    # Identificar columnas para clave única del hogar
    cols_hogar = [c for c in ['conglome', 'vivienda', 'hogar'] if c in df.columns]
    
    # Desduplicar para garantizar 1 fila = 1 hogar
    if cols_hogar:
        df_hogares = df.drop_duplicates(subset=cols_hogar)
    else:
        df_hogares = df  # Si la base ya viene estructurada a nivel hogar
    
    # Cálculo porcentual del 'result' por año
    conteo = df_hogares[col_result].value_counts(normalize=True, dropna=False) * 100
    df_pct = conteo.reset_index()
    df_pct.columns = ['categoria_result', 'porcentaje']
    df_pct['anio'] = anio
    resumen_resultados.append(df_pct)

if resumen_resultados:
    df_resumen = pd.concat(resumen_resultados, ignore_index=True)
    
    # Pivotear matriz temporal: Categorías vs Años
    cuadro_tiempo = df_resumen.pivot(
        index='categoria_result', 
        columns='anio', 
        values='porcentaje'
    ).fillna(0)
    
    # Guardar matriz consolidada
    ruta_cuadro_csv = os.path.join(RUTA_SALIDA, "porcentaje_no_respuesta_2019_2025.csv")
    cuadro_tiempo.to_csv(ruta_cuadro_csv, encoding='utf-8-sig')
    
    print("\nCUADRO PORCENTUAL DE RESULTADOS DE ENTREVISTA ('RESULT') 2019 - 2025:")
    print(cuadro_tiempo.round(2))
    print(f"\nMatriz temporal guardada en: {ruta_cuadro_csv}")

# ==============================================================================
# PUNTO 5: GUARDAR DATAFRAMES FILTRADOS POR CADA AÑO
# ==============================================================================
print("\n--- Ejecutando Punto 5: Exportación de DataFrames Filtrados ---")

# Crear subcarpeta específica para mantener el orden (opcional)
carpeta_filtrados = os.path.join(RUTA_SALIDA, "dataframes_filtrados")
os.makedirs(carpeta_filtrados, exist_ok=True)

for anio, df in sorted(dict_dataframes_filtrados.items()):
    nombre_archivo = f"modulo1_filtrado_{anio}.csv"
    ruta_archivo = os.path.join(carpeta_filtrados, nombre_archivo)
    
    # Exportar individualmente
    df.to_csv(ruta_archivo, index=False, encoding='utf-8-sig')
    print(f"Guardado año {anio}: {ruta_archivo} ({len(df):,} filas)")

print("\n¡Proceso completado exitosamente!")

--- Ejecutando Punto 4: Generación de Cuadro Temporal de No Respuesta ---

CUADRO PORCENTUAL DE RESULTADOS DE ENTREVISTA ('RESULT') 2019 - 2025:
anio               2019   2020   2021   2022   2023   2024   2025
categoria_result                                                 
1                 66.06  58.38  69.17  66.21  64.85  61.75  61.12
2                 12.73   6.18   9.51  11.33  11.51  13.57  14.44
3                  3.27   2.44   3.13   3.07   3.59   3.66   3.56
4                  0.52   0.83   0.82   0.80   0.60   0.55   0.66
5                  5.50   3.47   6.19   6.42   6.58   7.02   6.82
7                 11.92  28.70  11.18  12.17  12.87  13.45  13.40

Matriz temporal guardada en: D:/FUND. CIENCIA DE DATOS/CDD-2025/Resultados_Procesados\porcentaje_no_respuesta_2019_2025.csv

--- Ejecutando Punto 5: Exportación de DataFrames Filtrados ---
Guardado año 2019: D:/FUND. CIENCIA DE DATOS/CDD-2025/Resultados_Procesados\dataframes_filtrados\modulo1_filtrado_2019.csv (43,868 filas)

**TRANSFORMANDO LOS DATOS**

In [59]:
import os
import glob
import pandas as pd
import numpy as np

# ==============================================================================
# CONFIGURACIÓN DE RUTAS (Única sección que se modifica)
# ==============================================================================
DIR_INPUT = r"D:/FUND. CIENCIA DE DATOS/CDD-2025/Resultados_Procesados/dataframes_filtrados"
DIR_OUTPUT = r"D:/FUND. CIENCIA DE DATOS/CDD-2025/Resultados_Procesados/dataframes_transformados"

os.makedirs(DIR_OUTPUT, exist_ok=True)


# ==============================================================================
# FUNCIONES AUXILIARES DE TRANSFORMACIÓN METODOLÓGICA
# ==============================================================================

def procesar_transformaciones(df):
    """
    Aplica las transformaciones especificadas en los puntos 1 al 5.
    """
    df = df.copy()
    
    # Normalización de nombres de columnas
    df.columns = df.columns.str.lower().str.strip()

    # --------------------------------------------------------------------------
    # 1. Variable Dicotómica Target
    # 0 = Respuesta (Completa [1] o Incompleta [2])
    # 1 = No Respuesta (Categorías >= 3: Rechazo, Ausente, Desocupada, etc.)
    # --------------------------------------------------------------------------
    col_result = [c for c in df.columns if 'result' in c][0]
    df['target'] = np.where(df[col_result].isin([1, 2]), 0, 1)

    # --------------------------------------------------------------------------
    # 2. Departamento y Provincia a partir de Ubigeo (Código de 6 dígitos)
    # Ubigeo: DD (Dpto), PP (Prov), DI (Dist)
    # --------------------------------------------------------------------------
    if 'ubigeo' in df.columns:
        # Asegurar formato string de 6 dígitos con ceros a la izquierda
        ubigeo_str = df['ubigeo'].astype(str).str.zfill(6)
        df['departamento_cod'] = ubigeo_str.str[:2]
        df['provincia_cod'] = ubigeo_str.str[:4]
    else:
        df['departamento_cod'] = np.nan
        df['provincia_cod'] = np.nan

    # --------------------------------------------------------------------------
    # 3. Área Urbano y Rural a partir del Estrato
    # Categorías 1 a 5 = Urbano (1) | Categorías 6 a 8 = Rural (0)
    # --------------------------------------------------------------------------
    if 'estrato' in df.columns:
        df['estrato_num'] = pd.to_numeric(df['estrato'], errors='coerce')
        df['area_urbano_rural'] = np.where(df['estrato_num'] <= 5, 'Urbano', 'Rural')
        df['es_urbano'] = np.where(df['estrato_num'] <= 5, 1, 0)
    else:
        df['area_urbano_rural'] = np.nan
        df['es_urbano'] = np.nan

    # --------------------------------------------------------------------------
    # 4. Región Natural a partir de 'dominio'
    # Dominio INEI:
    # 1: Costa Norte, 2: Costa Centro, 3: Costa Sur
    # 4: Sierra Norte, 5: Sierra Centro, 6: Sierra Sur
    # 7: Selva, 8: Lima Metropolitana
    # --------------------------------------------------------------------------
    if 'dominio' in df.columns:
        df['dominio_num'] = pd.to_numeric(df['dominio'], errors='coerce')
        
        condiciones_region = [
            df['dominio_num'] == 8,
            df['dominio_num'].isin([1, 2, 3]),
            df['dominio_num'].isin([4, 5, 6]),
            df['dominio_num'] == 7
        ]
        etiquetas_region = ['Lima Metropolitana', 'Costa', 'Sierra', 'Selva']
        
        df['region_natural'] = np.select(condiciones_region, etiquetas_region, default='Sin Clasificar')
    else:
        df['region_natural'] = np.nan

    # --------------------------------------------------------------------------
    # 5. Variables Discretas (Mes, Trimestre y Codificación Categórica)
    # --------------------------------------------------------------------------
    if 'mes' in df.columns:
        df['mes_num'] = pd.to_numeric(df['mes'], errors='coerce')
        df['trimestre'] = np.ceil(df['mes_num'] / 3).astype('Int64')
    else:
        df['mes_num'] = np.nan
        df['trimestre'] = np.nan

    # Creación de factores/códigos discretos categóricos
    cols_categoricas = ['departamento_cod', 'provincia_cod', 'area_urbano_rural', 'region_natural', 'dominio']
    for col in cols_categoricas:
        if col in df.columns:
            df[f'{col}_cat'] = df[col].astype('category').cat.codes

    return df


def generar_estadisticas_agregadas(df, anio):
    """
    6. Genera un reporte comparativo formal de la tasa de no respuesta (target = 1)
    según las diferentes dimensiones geográficas y temporales.
    """
    print(f"\n" + "=" * 90)
    print(f"             REPORTE ESTADÍSTICO AGREGADO DE NO RESPUESTA - ENAHO {anio}")
    print("=" * 90)
    
    dimensiones = [
        ('area_urbano_rural', 'Área Residencial'),
        ('region_natural', 'Región Natural'),
        ('trimestre', 'Trimestre del Año'),
        ('departamento_cod', 'Departamento')
    ]
    
    lista_tablas = []
    
    for col, nombre_dim in dimensiones:
        if col in df.columns:
            resumen = df.groupby(col).agg(
                Total_Hogares=('target', 'count'),
                Respuestas=('target', lambda x: (x == 0).sum()),
                No_Respuestas=('target', lambda x: (x == 1).sum()),
                Tasa_No_Respuesta_Pct=('target', lambda x: (x == 1).mean() * 100)
            ).reset_index()
            
            resumen.rename(columns={col: 'Categoría'}, inplace=True)
            resumen['Dimensión'] = nombre_dim
            resumen['Año'] = anio
            lista_tablas.append(resumen)
            
            print(f"\n--- Tasa de No Respuesta por {nombre_dim} ({anio}) ---")
            print(resumen.to_string(index=False))
            
    df_consolidado_stats = pd.concat(lista_tablas, ignore_index=True)
    return df_consolidado_stats


# ==============================================================================
# EJECUCIÓN PRINCIPAL DEL PIPELINE
# ==============================================================================
def ejecutar_pipeline():
    archivos_m1 = glob.glob(os.path.join(DIR_INPUT, "*.csv"))
    
    # Filtrar únicamente los años 2024 y 2025
    archivos_2024_2025 = [f for f in archivos_m1 if '2024' in f or '2025' in f]
    
    if not archivos_2024_2025:
        print(f"No se encontraron archivos de 2024 o 2025 en la ruta: {DIR_INPUT}")
        return

    reportes_estadisticos = []

    for ruta_file in archivos_2024_2025:
        nombre_file = os.path.basename(ruta_file)
        anio = '2024' if '2024' in nombre_file else '2025'
        
        print(f"\nProcesando ENAHO {anio} ({nombre_file})...")
        df_original = pd.read_csv(ruta_file, low_memory=False)
        
        # 1 a 5. Aplicar Transformaciones
        df_transformado = procesar_transformaciones(df_original)
        
        # 6. Elaborar Estadísticas Agregadas
        df_stats = generar_estadisticas_agregadas(df_transformado, anio)
        reportes_estadisticos.append(df_stats)
        
        # 7. Guarda DataFrames Transformados
        ruta_salida_df = os.path.join(DIR_OUTPUT, f"enaho_transformado_{anio}.csv")
        df_transformado.to_csv(ruta_salida_df, index=False, encoding='utf-8-sig')
        print(f"-> DataFrame transformado guardado exitosamente: {ruta_salida_df}")

    # Exportación consolidada de las estadísticas agregadas comparativas
    if reportes_estadisticos:
        df_reporte_final = pd.concat(reportes_estadisticos, ignore_index=True)
        ruta_stats_csv = os.path.join(DIR_OUTPUT, "reporte_estadistico_no_respuesta_2024_2025.csv")
        df_reporte_final.to_csv(ruta_stats_csv, index=False, encoding='utf-8-sig')
        print(f"\n" + "=" * 90)
        print(f"Reporte estadístico consolidado guardado en: {ruta_stats_csv}")
        print("=" * 90)


if __name__ == "__main__":
    ejecutar_pipeline()


Procesando ENAHO 2025 (modulo1_filtrado_2019.csv)...

             REPORTE ESTADÍSTICO AGREGADO DE NO RESPUESTA - ENAHO 2025

--- Tasa de No Respuesta por Área Residencial (2025) ---
Categoría  Total_Hogares  Respuestas  No_Respuestas  Tasa_No_Respuesta_Pct        Dimensión  Año
    Rural          16557       13072           3485              21.048499 Área Residencial 2025
   Urbano          27311       21493           5818              21.302772 Área Residencial 2025

--- Tasa de No Respuesta por Región Natural (2025) ---
         Categoría  Total_Hogares  Respuestas  No_Respuestas  Tasa_No_Respuesta_Pct      Dimensión  Año
             Costa          12320       10236           2084              16.915584 Región Natural 2025
Lima Metropolitana           5441        4030           1411              25.932733 Región Natural 2025
             Selva           8845        6990           1855              20.972301 Región Natural 2025
            Sierra          17262       13309        